In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install -U wandb sacrebleu
!pip -q install -U bitsandbytes transformers peft accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 144.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.1 MB/s eta 0:00:00:00:0100:01


# 1.学習コード

## 1-1.Config

In [3]:
import os, re, math, random
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Mistral-7B-Instruct-v0.3
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

TRAIN_CSV  = "/content/drive/MyDrive/kaggle/Akkadian/data/train.csv"
OUTPUT_DIR = "/content/drive/MyDrive/kaggle/Akkadian/models/mistral-7b-lora-deeppast-v1"

SEED      = 42
VAL_RATIO = 0.02

MAX_LEN    = 1024
MICRO_BS   = 1
GRAD_ACCUM = 16

WARMUP_RATIO = 0.03
SAVE_STEPS   = 10
EVAL_STEPS   = 10
LOG_STEPS    = 10

# Mistral-7B は大きいので 4bit QLoRA を推奨（Colab T4/A100 共通）
USE_4BIT = True

# LoRA 設定（Mistral の線形レイヤーに合わせた定番 modules）
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LR     = 2e-4
EPOCHS = 2

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

## 1-2 wandbの設定

In [4]:
import wandb
wandb.login()

RUN_NAME = "mistral-7b-lora-deeppast-v1"
wandb.init(
    project="deep-past-akkadian",
    name=RUN_NAME,
    config={
        "model": MODEL_NAME,
        "seed": SEED,
        "val_ratio": VAL_RATIO,
        "max_len": MAX_LEN,
        "micro_bs": MICRO_BS,
        "grad_accum": GRAD_ACCUM,
        "epochs": EPOCHS,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "use_4bit": USE_4BIT,
    },
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuzurihainori315 (yuzurihainori315-national-institute-of-technology-ibarak) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1-3 Helper

In [6]:
def normalize_gaps(text):
    """Normalize ALL gap variants to a single <gap> token."""
    if not isinstance(text, str):
        return str(text) if text else ""

    text = re.sub(r'\((?:large\s+)?break\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\(\d+\s+broken\s+lines?\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\.{3,}', ' <gap> ', text)
    text = re.sub(r'……', ' <gap> ', text)
    text = re.sub(r'…', ' <gap> ', text)
    text = re.sub(r'\[x\]', '<gap>', text, flags=re.I)
    text = re.sub(r'\bxx+\b', ' <gap> ', text)
    text = re.sub(r'\bx\b', '<gap>', text)
    return text


def normalize_transliteration(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = normalize_gaps(text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')

    subscript_to_normal = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(subscript_to_normal)

    text = text.replace('KÙ.B.', 'KÙ.BABBAR')
    text = re.sub(r'(\d+\.\d{4})\d+', r'\1', text)

    for pat, frac in [
        (r'\b0\.8333\d*\b', '⅚'), (r'\b0\.1666\d*\b', '⅙'),
        (r'\b0\.6666\d*\b', '⅔'), (r'\b0\.3333\d*\b', '⅓'),
        (r'\b0\.625\b', '⅝'),
        (r'\b0\.75\b', '¾'), (r'\b0\.25\b', '¼'), (r'\b0\.5\b', '½'),
    ]:
        text = re.sub(pat, frac, text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text


def postprocess_output(text):
    if not isinstance(text, str):
        return str(text) if text else "broken text"
    if not text.strip():
        return "broken text"

    text = re.sub(r'\[x\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\(x\)', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'(\.{3,}|…|\[\.+\])', '<gap>', text)
    text = re.sub(r'\((?:large\s+)?break\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\(\d+\s+broken\s+lines?\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\bx\b', '<gap>', text)

    text = re.sub(r'\(fem\.?\s*(?:plur\.?|pl\.?|sing\.?|singular|plural)?\)', '', text, flags=re.I)
    text = re.sub(r'\((?:plur|pl|sing|singular|plural)\.?\)', '', text, flags=re.I)
    text = re.sub(r'\(\?\)', '', text)
    text = re.sub(r'\(!\)', '', text)
    text = re.sub(r'\bfem\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bsing\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bpl\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bplural\b\s*', '', text, flags=re.I)

    text = re.sub(r'\(d\)', '{d}', text)
    text = re.sub(r'\(ki\)', '{ki}', text)
    text = re.sub(r'\(TÚG\)', 'TÚG', text)

    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('-gold', 'pašallum gold')
    text = text.replace('-tax', 'šadduātum tax')
    text = re.sub(r'(?<!kutānum )\btextiles\b', 'kutānum textiles', text)

    text = re.sub(r'\b1\s*/\s*12\s*\(?\s*shekel\s*\)?', '⅔ shekel 15 grains', text, flags=re.I)
    text = re.sub(r'\b5\s+11\s*/\s*12\s+shekels?', '6 shekels less 15 grains', text, flags=re.I)
    text = re.sub(r'\b7\s*/\s*12\s+shekels?', '½ shekel 15 grains', text, flags=re.I)
    text = re.sub(r'\b5\s*/\s*12\s+shekels?', '15 grains', text, flags=re.I)
    text = re.sub(r'(\w+)\s+/\s+(\w+)', r'\1', text)

    for roman, integer in [
        ('XII', '12'), ('VIII', '8'), ('XI', '11'), ('VII', '7'),
        ('VI', '6'), ('IX', '9'), ('IV', '4'), ('III', '3'),
        ('II', '2'), ('X', '10'), ('V', '5'), ('I', '1'),
    ]:
        text = re.sub(r'\b(month)\s+' + roman + r'\b', r'\1 ' + integer, text, flags=re.I)

    text = re.sub(r'(\d+\.\d{4})\d+', r'\1', text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')

    subscript_map = {'₀':'0','₁':'1','₂':'2','₃':'3','₄':'4','₅':'5','₆':'6','₇':'7','₈':'8','₉':'9'}
    for sub, normal in subscript_map.items():
        text = text.replace(sub, normal)

    text = text.replace('<gap>', '\x00GAP\x00')
    forbidden_chars = '()—–<>⌈⌊⌋[]+ʾ/;'
    for char in forbidden_chars:
        text = text.replace(char, '')
    text = re.sub(r'<<\s*>>', '', text)
    text = text.replace('\x00GAP\x00', '<gap>')

    for pattern, replacement in [
        (r'(\d+)\.8333\d*\b', r'\1 ⅚'), (r'\b0\.8333\d*\b', '⅚'),
        (r'(\d+)\.1666\d*\b', r'\1 ⅙'), (r'\b0\.1666\d*\b', '⅙'),
        (r'(\d+)\.6666\d*\b', r'\1 ⅔'), (r'\b0\.6666\d*\b', '⅔'),
        (r'(\d+)\.3333\d*\b', r'\1 ⅓'), (r'\b0\.3333\d*\b', '⅓'),
        (r'(\d+)\.625\b', r'\1 ⅝'), (r'\b0\.625\b', '⅝'),
        (r'(\d+)\.75\b', r'\1 ¾'), (r'\b0\.75\b', '¾'),
        (r'(\d+)\.25\b', r'\1 ¼'), (r'\b0\.25\b', '¼'),
        (r'(\d+)\.5\b', r'\1 ½'), (r'\b0\.5\b', '½'),
    ]:
        text = re.sub(pattern, replacement, text)

    text = text.replace(' 1 2', ' ½').replace(' 1 3', ' ⅓').replace(' 2 3', ' ⅔')
    text = text.replace(' 1 4', ' ¼').replace(' 3 4', ' ¾')

    text = re.sub(r'\.\..+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^-\s*', '', text)
    text = re.sub(r'\s*-$', '', text)

    return text if text else "broken text"

## 1-4 学習

In [ ]:
# ---------- Load tokenizer ----------
# Mistral は AutoTokenizer のみで十分（AutoProcessor 不要）
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "right"          # 学習時は right-padding が安定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------- Load model ----------
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


# ---------- Load train.csv -> train/val split ----------
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_val  = max(1, int(len(df) * VAL_RATIO))
df_val = df.iloc[:n_val].copy()
df_trn = df.iloc[n_val:].copy()

train_ds = Dataset.from_pandas(df_trn, preserve_index=False)
val_ds   = Dataset.from_pandas(df_val, preserve_index=False)


# ---------- Tokenize with chat_template + completion-only loss ----------
def build_texts(translit: str, translation: str):
    """
    Mistral の chat_template は [INST] … [/INST] 形式。
    apply_chat_template が自動的にフォーマットする。
    """
    user      = f"Translate Akkadian to English: {normalize_transliteration(translit)}"
    assistant = str(translation).strip().replace("\n", " ")

    messages  = [{"role": "user", "content": user},
                 {"role": "assistant", "content": assistant}]
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    prompt_only = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt_only, full_text


def tokenize_example(ex):
    prompt_text, full_text = build_texts(ex["transliteration"], ex["translation"])

    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    input_ids = full["input_ids"]
    attn      = full["attention_mask"]

    boundary = len(prompt_ids)
    if len(input_ids) < boundary or input_ids[:boundary] != prompt_ids:
        boundary = None
        max_search = min(256, len(input_ids))
        for b in range(max_search):
            if input_ids[b : b + len(prompt_ids)] == prompt_ids:
                boundary = b + len(prompt_ids)
                break
        if boundary is None:
            boundary = min(len(prompt_ids), len(input_ids))

    labels = [-100] * boundary + input_ids[boundary:]
    labels = labels[: len(input_ids)]

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}


train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_example,   remove_columns=val_ds.column_names)


@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any
    pad_to_multiple_of: int = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
             for f in features],
            padding=True,
            return_tensors="pt",
            pad_to_multiple_of=self.pad_to_multiple_of,
        )
        max_len = batch["input_ids"].shape[1]
        labels  = []
        for f in features:
            l = f["labels"] + [-100] * (max_len - len(f["labels"]))
            labels.append(l)
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

collator = DataCollatorForCausalLM(tokenizer)


# ---------- Train ----------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=MICRO_BS,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",

    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="wandb",

    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()

# LoRA アダプタ保存
trainer.model.save_pretrained(os.path.join(OUTPUT_DIR, "adapter"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "adapter"))

# ---------- Merge LoRA into base and save ----------
from peft import PeftModel

base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

merged = PeftModel.from_pretrained(base_for_merge, os.path.join(OUTPUT_DIR, "adapter"))
merged = merged.merge_and_unload()

MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")
os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("Saved:")
print(" - adapter:", os.path.join(OUTPUT_DIR, "adapter"))
print(" - merged :", MERGED_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,0.998006,0.833209
2,0.669978,0.752859


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved:
 - adapter: /content/drive/MyDrive/kaggle/Akkadian/models/mistral-7b-lora-deeppast-v1/adapter
 - merged : /content/drive/MyDrive/kaggle/Akkadian/models/mistral-7b-lora-deeppast-v1/merged


## 1-5 Validation Score (BLEU / chrF++ Geometric Mean)

In [8]:
from sacrebleu.metrics import BLEU, CHRF
from tqdm import tqdm

# ---- Load merged model from disk (same as section-2 inference) ----
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")

# ---------- Load train.csv -> train/val split ----------
print("Loading and preparing validation data …")
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_val = max(1, int(len(df) * VAL_RATIO))
df_val = df.iloc[:n_val].copy()
df_trn = df.iloc[n_val:].copy()

train_ds = Dataset.from_pandas(df_trn, preserve_index=False)
val_ds = Dataset.from_pandas(df_val, preserve_index=False)


print("--- Loading merged model for evaluation ---")
eval_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
eval_tokenizer.padding_side = "left"
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
eval_model.eval()

EVAL_BATCH_SIZE = 4   # 7B モデルなので Qwen より小さめに

# Mistral の stop token は </s>（eos_token）のみ
_stop_ids = [eval_tokenizer.eos_token_id]


def _translate_batch(translit_list):
    """Return translated strings for a batch of transliterations."""
    messages_batch = [
        [{"role": "user",
          "content": f"Translate Akkadian to English: {normalize_transliteration(t)}"}]
        for t in translit_list
    ]
    raw_prompts = [
        eval_tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    inputs = eval_tokenizer(
        raw_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
    ).to(eval_model.device)

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=_stop_ids,
        )

    input_len = inputs.input_ids.shape[1]
    decoded   = eval_tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    return [postprocess_output(t.strip().replace("\n", " ")) for t in decoded]


print("Running inference on validation set …")
val_translits  = df_val["transliteration"].tolist()
val_references = (
    df_val["translation"].astype(str).str.strip().str.replace("\n", " ").tolist()
)

val_predictions = []
for i in tqdm(range(0, len(val_translits), EVAL_BATCH_SIZE)):
    val_predictions.extend(_translate_batch(val_translits[i : i + EVAL_BATCH_SIZE]))

# ----- BLEU -----
bleu_metric = BLEU(effective_order=True)
bleu_score  = bleu_metric.corpus_score(val_predictions, [val_references]).score

# ----- chrF++ -----
chrf_metric = CHRF(word_order=2)
chrf_score  = chrf_metric.corpus_score(val_predictions, [val_references]).score

# ----- Geometric Mean -----
geo_mean = math.sqrt(bleu_score * chrf_score)

print("\n========== Validation Scores ==========")
print(f"  BLEU   : {bleu_score:.4f}")
print(f"  chrF++ : {chrf_score:.4f}")
print(f"  GeoMean: {geo_mean:.4f}")
print("========================================")

try:
    wandb.log({
        "val/bleu":     bleu_score,
        "val/chrf++":   chrf_score,
        "val/geo_mean": geo_mean,
    })
except Exception:
    pass

Loading and preparing validation data …
--- Loading merged model for evaluation ---


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Running inference on validation set …


100%|██████████| 8/8 [01:46<00:00, 13.33s/it]


========== Validation Scores ==========
  BLEU   : 26.6382
  chrF++ : 51.0168
  GeoMean: 36.8646


# 2.推論

In [9]:
import os
import pandas as pd
import torch
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# =========================================================
# Configuration
# =========================================================
MODEL_DIR  = "/content/drive/MyDrive/kaggle/Akkadian/models/mistral-7b-lora-deeppast-v1/merged"
TEST_CSV   = "/content/drive/MyDrive/kaggle/Akkadian/data/test.csv"
OUTPUT_CSV = "/content/drive/MyDrive/kaggle/Akkadian/results/submission_mistral.csv"
BATCH_SIZE = 4   # 7B モデルなので控えめに

In [10]:
# =========================================================
# 1) Load Test Data
# =========================================================
print("--- Loading Test Data ---")
test_df = pd.read_csv(TEST_CSV)

# =========================================================
# 2) Initialize Tokenizer
# =========================================================
print("--- Initializing Tokenizer ---")
tokenizer_infer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
tokenizer_infer.padding_side = "left"
if tokenizer_infer.pad_token is None:
    tokenizer_infer.pad_token = tokenizer_infer.eos_token

# =========================================================
# 3) Build prompts with chat template
# =========================================================
messages_list = [
    [{"role": "user",
      "content": f"Translate Akkadian to English: {normalize_transliteration(text)}"}]
    for text in test_df["transliteration"]
]

raw_prompts = [
    tokenizer_infer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
    for m in messages_list
]

# =========================================================
# 4) Load merged model
# =========================================================
print("--- Loading Merged Model ---")
model_infer = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model_infer.eval()

# =========================================================
# 5) Stop token IDs
# =========================================================
stop_ids = [tokenizer_infer.eos_token_id]
print(f"Stopping on token IDs: {stop_ids}")

# =========================================================
# 6) Inference loop
# =========================================================
predictions = []

for i in tqdm(range(0, len(raw_prompts), BATCH_SIZE)):
    batch_prompts = raw_prompts[i : i + BATCH_SIZE]

    inputs = tokenizer_infer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(model_infer.device)

    with torch.no_grad():
        outputs = model_infer.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=tokenizer_infer.pad_token_id,
            eos_token_id=stop_ids,
        )

    input_len        = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_len:]
    decoded_batch    = tokenizer_infer.batch_decode(generated_tokens, skip_special_tokens=True)
    predictions.extend(decoded_batch)

# =========================================================
# 7) Postprocess
# =========================================================
final_predictions = [
    postprocess_output(text.strip().replace("\n", " "))
    for text in predictions
]

# =========================================================
# 8) Save submission.csv
# =========================================================
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
submission_df = pd.DataFrame({"id": test_df["id"], "translation": final_predictions})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"Submission saved to {OUTPUT_CSV}")
submission_df.head()

--- Loading Test Data ---
--- Initializing Tokenizer ---
--- Loading Merged Model ---


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Stopping on token IDs: [2]


100%|██████████| 1/1 [00:02<00:00,  2.63s/it]

Submission saved to /content/drive/MyDrive/kaggle/Akkadian/results/submission_mistral.csv


,id,translation
0,0,"Thus the Kanesh colony, say to Aqilī, the smit..."
1,1,When the Narrow Track is open on the way to th...
2,2,"As we have heard your orders there, whether it..."
3,3,Our envoys must travel overland and by river t...
